This notebook provides examples to go along with the [textbook](https://underactuated.csail.mit.edu/acrobot.html).  I recommend having both windows open, side-by-side!


In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "underactuated[all]>=2026.9.8"

In [ ]:
import numpy as np
from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    ControllabilityMatrix,
    DiagramBuilder,
    LeafSystem,
    Linearize,
    LinearQuadraticRegulator,
    MeshcatVisualizer,
    Parser,
    Simulator,
)

from underactuated import ConfigureParser, running_as_notebook
from underactuated.meshcat_utils import MeshcatSliders, StartMeshcat

In [ ]:
# Start the visualizer (run this cell only once, each instance consumes a port)
meshcat = StartMeshcat()

# Dynamics of the Cart-Pole

In [ ]:
# TODO(russt): Generalize this to a generic gamepad system.


class GamepadTeleop(LeafSystem):
    def __init__(self, meshcat):
        LeafSystem.__init__(self)
        self._meshcat = meshcat
        port = self.DeclareVectorOutputPort("left_stick_x", 1, self.OutputLeftStick)
        # The gamepad has undeclared state.  For now, we accept it,
        # and simply disable caching on the output port.
        port.disable_caching_by_default()

    def CreateStickDeadzone(self, x, y):
        stick = np.array([x, y])
        deadzone = 0.3
        m = np.linalg.norm(stick)
        if m < deadzone:
            return np.array([0, 0])
        over = (m - deadzone) / (1 - deadzone)
        return stick * over / m

    def OutputLeftStick(self, context, output):
        gamepad = self._meshcat.GetGamepad()

        # https://beej.us/blog/data/javascript-gamepad/

        left = self.CreateStickDeadzone(gamepad.axes[0], gamepad.axes[1])
        output[0] = 10 * left[0]


def cartpole_demo():
    builder = DiagramBuilder()
    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/cartpole.urdf")
    plant.Finalize()

    meshcat.Delete()
    meshcat.Set2dRenderMode(xmin=-2.5, xmax=2.5, ymin=-1.0, ymax=2.5)
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

    if meshcat.GetGamepad().index != None:
        print("Gamepad teleop enabled. Use the left stick to apply force to the cart.")
        gamepad_teleop = builder.AddSystem(GamepadTeleop(meshcat))
        builder.Connect(
            gamepad_teleop.get_output_port(), plant.get_actuation_input_port()
        )
    else:
        print("Use the slider in the MeshCat controls to apply force to the cart.")
        meshcat.AddSlider(
            "u",
            min=-5,
            max=5,
            step=0.1,
            value=0.0,
            decrement_keycode="ArrowLeft",
            increment_keycode="ArrowRight",
        )
        force_system = builder.AddSystem(MeshcatSliders(meshcat, ["u"]))
        builder.Connect(
            force_system.get_output_port(), plant.get_actuation_input_port()
        )

    diagram = builder.Build()

    # Set up a simulator to run this diagram
    simulator = Simulator(diagram)
    context = simulator.get_mutable_context()

    # Set the initial conditions (x, theta, xdot, thetadot)
    context.SetContinuousState([0, 1, 0, 0])

    if running_as_notebook:  # Then we're not just running as a test on CI.
        simulator.set_target_realtime_rate(1.0)

        print("Press 'Stop Simulation' in MeshCat to continue.")
        meshcat.AddButton("Stop Simulation", "Escape")
        while meshcat.GetButtonClicks("Stop Simulation") < 1:
            simulator.AdvanceTo(simulator.get_context().get_time() + 1.0)
    else:
        simulator.AdvanceTo(0.1)

    meshcat.DeleteAddedControls()


cartpole_demo()

# LQR for the Cart-Pole

In [ ]:
def cartpole_balancing_example():
    def UprightState():
        state = (0, np.pi, 0, 0)
        return state

    def Controllability(plant):
        context = plant.CreateDefaultContext()
        plant.get_actuation_input_port().FixValue(context, [0])
        plant.SetPositionsAndVelocities(context, UprightState())

        linearized_plant = Linearize(
            plant,
            context,
            input_port_index=plant.get_actuation_input_port().get_index(),
            output_port_index=plant.get_state_output_port().get_index(),
        )
        print(linearized_plant.A())
        print(linearized_plant.B())
        print(
            f"The singular values of the controllability matrix are: {np.linalg.svd(ControllabilityMatrix(linearized_plant), compute_uv=False)}"
        )

    def BalancingLQR(plant):
        # Design an LQR controller for stabilizing the CartPole around the upright.
        # Returns a (static) AffineSystem that implements the controller (in
        # the original CartPole coordinates).

        context = plant.CreateDefaultContext()
        plant.get_actuation_input_port().FixValue(context, [0])

        plant.SetPositionsAndVelocities(context, UprightState())

        Q = np.diag((10.0, 10.0, 1.0, 1.0))
        R = np.array([1])

        # MultibodyPlant has many (optional) input ports, so we must pass the
        # input_port_index to LQR.
        return LinearQuadraticRegulator(
            plant,
            context,
            Q,
            R,
            input_port_index=plant.get_actuation_input_port().get_index(),
        )

    builder = DiagramBuilder()
    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/cartpole.urdf")
    plant.Finalize()

    controller = builder.AddSystem(BalancingLQR(plant))
    builder.Connect(plant.get_state_output_port(), controller.get_input_port(0))
    builder.Connect(controller.get_output_port(0), plant.get_actuation_input_port())

    # Setup visualization
    meshcat.Delete()
    meshcat.Set2dRenderMode(xmin=-2.5, xmax=2.5, ymin=-1.0, ymax=2.5)
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

    diagram = builder.Build()

    # Set up a simulator to run this diagram
    simulator = Simulator(diagram)
    context = simulator.get_mutable_context()
    plant_context = plant.GetMyMutableContextFromRoot(context)

    # Simulate
    simulator.set_target_realtime_rate(1.0 if running_as_notebook else 0.0)
    duration = 5.0 if running_as_notebook else 0.1
    for i in range(5):
        context.SetTime(0.0)
        plant.SetPositionsAndVelocities(
            plant_context,
            UprightState()
            + 0.1
            * np.random.randn(
                4,
            ),
        )
        simulator.Initialize()
        simulator.AdvanceTo(duration)


np.set_printoptions(formatter={"float": lambda x: "{0:0.4f}".format(x)})
cartpole_balancing_example()

# 作业分析（在课程原代码之后新增）

本文件是 `Exp3'2-cartpole.ipynb` 的**副本**：上方课程原始 cell 保持不变，下面追加作业要求的运行结果解读与参数敏感性分析。

- 原文件未被修改，可随时回退；
- 下面的 cell 只依赖 Drake，不依赖上方的 MeshCat 窗口，可单独运行；
- 结果与 `Exp3'2-cartpole-作业解答.md`、`exp3_2_analysis/cp_analysis.py` 一致。

> 若点击 “Run All”，上方 cell 5 会等待 MeshCat 页面里的 “Stop Simulation” 按钮，这是原代码的交互设计。


In [ ]:

# ===== 本节公共函数（独立于上方课程代码）=====
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt
from pydrake.all import (
    AddMultibodyPlantSceneGraph, ControllabilityMatrix, DiagramBuilder,
    Linearize, LinearQuadraticRegulator, Parser, Saturation, Simulator,
)
from pydrake.multibody.plant import MultibodyPlant
from pydrake.multibody.tree import RotationalInertia, SpatialInertia
from pydrake.systems.primitives import LogVectorOutput
from underactuated import ConfigureParser

%matplotlib inline
plt.rcParams.update({
    "font.sans-serif": ["Arial Unicode MS", "Hiragino Sans GB"],
    "axes.unicode_minus": False,
    "figure.dpi": 120,
})
np.set_printoptions(precision=4, suppress=True, linewidth=180)

XSTAR = np.array([0.0, np.pi, 0.0, 0.0])   # [x, theta, xdot, thetadot]
BASE_Q = np.diag([10.0, 10.0, 1.0, 1.0])
BASE_R = np.array([1.0])
MC, MP, LC = 10.0, 1.0, 0.5
T = 5.0
ANGLE_TOL, VEL_TOL = 0.05, 0.2
CART_TOL, CART_VEL_TOL = 0.5, 0.5
MODEL_URL = "package://underactuated/models/cartpole.urdf"


def c_wrap(a):
    return (a + np.pi) % (2.0 * np.pi) - np.pi


def c_make_plant(builder=None, gravity=9.81):
    if builder is None:
        plant = MultibodyPlant(0.0)
        sg = None
    else:
        plant, sg = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl(MODEL_URL)
    plant.Finalize()
    plant.mutable_gravity_field().set_gravity_vector(np.array([0.0, 0.0, -gravity]))
    return plant, sg


def c_set_params(plant, ctx, mc=MC, mp=MP, lc=LC):
    plant.GetBodyByName("cart").SetMass(ctx, mc)
    M = SpatialInertia.MakeFromCentralInertia(
        mp, np.array([0.0, 0.0, -lc]), RotationalInertia(0, 0, 0))
    plant.GetBodyByName("pole").SetSpatialInertiaInBodyFrame(ctx, M)


def c_ctx(plant, mc=MC, mp=MP, lc=LC):
    ctx = plant.CreateDefaultContext()
    plant.get_actuation_input_port().FixValue(ctx, [0.0])
    plant.SetPositionsAndVelocities(ctx, XSTAR)
    c_set_params(plant, ctx, mc, mp, lc)
    return ctx


def c_linear(mc=MC, mp=MP, lc=LC, gravity=9.81):
    plant, _ = c_make_plant(gravity=gravity)
    ctx = c_ctx(plant, mc, mp, lc)
    lin = Linearize(
        plant, ctx,
        input_port_index=plant.get_actuation_input_port().get_index(),
        output_port_index=plant.get_state_output_port().get_index())
    A, B = lin.A(), lin.B()
    Cm = ControllabilityMatrix(lin).astype(float)
    sv = np.linalg.svd(Cm, compute_uv=False)
    lam = np.linalg.eigvals(A)
    pbh = [int(np.linalg.matrix_rank(np.hstack([li * np.eye(4) - A, B])))
           for li in {round(v.real, 8) + 1j * round(v.imag, 8) for v in lam}]
    zeros = {}
    E = np.block([[np.eye(4), np.zeros((4, 1))],
                  [np.zeros((1, 4)), np.zeros((1, 1))]])
    for name, cout in (("u->x", np.array([[1.0, 0, 0, 0]])),
                       ("u->θ", np.array([[0.0, 1, 0, 0]]))):
        M = np.block([[A, B], [cout, np.zeros((1, 1))]])
        z = sla.eigvals(M, E)
        zeros[name] = sorted(float(np.real(v)) for v in z if np.isfinite(v))
    return dict(A=A, B=B, Cm=Cm, sv=sv, eig=lam,
                rank=int(np.linalg.matrix_rank(Cm)), pbh=pbh, zeros=zeros)


def c_design(Q=BASE_Q, R=BASE_R, mc=MC, mp=MP, lc=LC, gravity=9.81):
    plant, _ = c_make_plant(gravity=gravity)
    ctx = c_ctx(plant, mc, mp, lc)
    return LinearQuadraticRegulator(
        plant, ctx, Q, R,
        input_port_index=plant.get_actuation_input_port().get_index())


def c_build(controller, mc=MC, mp=MP, lc=LC, lim=1e6):
    builder = DiagramBuilder()
    plant, _ = c_make_plant(builder)
    ctrl = builder.AddSystem(controller)
    sat = builder.AddSystem(Saturation([-lim], [lim]))
    builder.Connect(plant.get_state_output_port(), ctrl.get_input_port(0))
    builder.Connect(ctrl.get_output_port(0), sat.get_input_port(0))
    builder.Connect(sat.get_output_port(0), plant.get_actuation_input_port())
    log_x = LogVectorOutput(plant.get_state_output_port(), builder, 0.01)
    log_u = LogVectorOutput(sat.get_output_port(), builder, 0.01)
    log_ucmd = LogVectorOutput(ctrl.get_output_port(0), builder, 0.01)
    diagram = builder.Build()
    return dict(diagram=diagram, plant=plant, log_x=log_x, log_u=log_u,
                log_ucmd=log_ucmd, D=controller.D().ravel())


def c_sim(diag, x0, mc=MC, mp=MP, lc=LC, T=T):
    sim = Simulator(diag["diagram"])
    ctx = sim.get_mutable_context()
    plant = diag["plant"]
    pctx = plant.GetMyMutableContextFromRoot(ctx)
    c_set_params(plant, pctx, mc, mp, lc)
    plant.SetPositionsAndVelocities(pctx, np.asarray(x0, float))
    sim.Initialize()
    sim.AdvanceTo(T)
    return (diag["log_x"].FindLog(ctx).sample_times().copy(),
            diag["log_x"].FindLog(ctx).data().copy(),
            diag["log_u"].FindLog(ctx).data().copy(),
            diag["log_ucmd"].FindLog(ctx).data().copy())


def c_metrics(ts, X, U, Uc):
    x, th, xd, thd = X[0], c_wrap(X[1] - np.pi), X[2], X[3]
    pole_ok = bool(abs(th[-1]) < ANGLE_TOL and abs(thd[-1]) < VEL_TOL)
    cart_ok = bool(abs(x[-1]) < CART_TOL and abs(xd[-1]) < CART_VEL_TOL)
    success = pole_ok and cart_ok
    bad = (np.abs(th) > ANGLE_TOL) | (np.abs(thd) > VEL_TOL)
    settle = ((float(ts[np.where(bad)[0][-1] + 1]) if bad.any() else 0.0)
              if pole_ok else np.nan)
    early = ts < 0.5
    return dict(success=success, pole_ok=pole_ok, cart_ok=cart_ok, settle=settle,
                final_x=float(abs(x[-1])), max_x=float(np.max(np.abs(x))),
                peak_ucmd_early=float(np.max(np.abs(Uc[0][early]))),
                max_force=float(np.max(np.abs(U[0]))))


def c_run(mc=MC, mp=MP, lc=LC, Q=BASE_Q, R=BASE_R, lim=1e6, x0s=None):
    ctrl = c_design(Q, R, mc, mp, lc)
    diag = c_build(ctrl, mc, mp, lc, lim)
    rows = [c_metrics(*c_sim(diag, x0, mc, mp, lc)) for x0 in x0s]
    return rows, diag


def c_summary(rows):
    ok = [r for r in rows if r["pole_ok"]]
    return dict(n=len(rows), success=int(sum(r["success"] for r in rows)),
                rate=float(np.mean([r["success"] for r in rows])),
                pole_rate=float(np.mean([r["pole_ok"] for r in rows])),
                cart_rate=float(np.mean([r["cart_ok"] for r in rows])),
                peak=float(np.median([r["peak_ucmd_early"] for r in rows])),
                peak90=float(np.percentile([r["peak_ucmd_early"] for r in rows], 90)),
                max_x=float(np.median([r["max_x"] for r in rows])),
                final_x=float(np.median([r["final_x"] for r in rows])),
                settle=(float(np.nanmedian([r["settle"] for r in ok])) if ok else np.nan))


## 1. 线性化与可控性

计算 $A,B$、开环特征值、能控性矩阵奇异值、右半平面零点和 LQR 设计结果，并与讲义归一化模型对照。

In [ ]:

ld = c_linear()
ctrl = c_design()
D = ctrl.D().ravel()
A, B = ld["A"], ld["B"]
cl = np.linalg.eigvals(A + B @ D.reshape(1, -1))

print("URDF 默认参数：mc=10 kg, mp=1 kg, pole COM=0.5 m, g=9.81\n")
print("A =\n", A)
print("B =", B.ravel())
print("开环特征值:", np.sort_complex(ld["eig"]))
print("rank(C) =", ld["rank"], " PBH ranks =", ld["pbh"])
print("σ(C) =", ld["sv"], " 条件数 =", ld["sv"][0] / ld["sv"][-1])
print("零点:", ld["zeros"])
print("\nLQR: u = D·(x - x*)，D =", D, " y0 =", float(ctrl.y0()[0]))
print("闭环极点:", np.sort_complex(cl))

# 与讲义归一化模型对照：mc=mp=1, lc=1, g=1
ldn = c_linear(mc=1.0, mp=1.0, lc=1.0, gravity=1.0)
ctrln = c_design(mc=1.0, mp=1.0, lc=1.0, gravity=1.0)
Kln = -ctrln.D().ravel()
cln = np.linalg.eigvals(ldn["A"] + ldn["B"] @ (-Kln).reshape(1, -1))
print("\n讲义归一化模型（mc=mp=1, lc=1, g=1）：")
print("A =\n", ldn["A"])
print("B =", ldn["B"].ravel())
print("开环特征值:", np.sort_complex(ldn["eig"]))
print("zeros:", ldn["zeros"])
print("K = -D =", Kln, " 闭环极点:", np.sort_complex(cln))


## 2. LQR 镇定与基线统计

与 notebook cell 7 相同的 `Q`、`R`、初始扰动 $0.1\mathcal N(0,I)$ 和 5 s 时域；分别统计“摆杆平衡”和“摆杆+小车回到容差”。

In [ ]:

# 与 notebook 相同：x0 = x* + 0.1*N(0,I4)，T = 5 s，无限幅
rng = np.random.default_rng(42)
x0s100 = [XSTAR + 0.1 * rng.standard_normal(4) for _ in range(100)]

rng5 = np.random.default_rng(7)
rows5, _ = c_run(x0s=[XSTAR + 0.1 * rng5.standard_normal(4) for _ in range(5)])
s5 = c_summary(rows5)
print(f"notebook 5 次试验复现：摆杆平衡 {int(s5['pole_rate']*5)}/5，"
      f"完整状态成功 {s5['success']}/5，峰值推力中位数 {s5['peak']:.1f} N")

rows, _ = c_run(x0s=x0s100)
s = c_summary(rows)
print(f"\n100 次随机试验（无输入限幅）：")
print(f"  摆杆平衡成功率      {s['pole_rate']:.0%}")
print(f"  摆杆+小车回到容差  {s['rate']:.0%}  (容差: |x|<0.5 m, |xdot|<0.5 m/s)")
print(f"  峰值推力: 中位 {s['peak']:.1f} N, 90 分位 {s['peak90']:.1f} N, "
      f"最大 {max(r['max_force'] for r in rows):.1f} N")
print(f"  小车最大偏离中位数 {s['max_x']:.2f} m；5 s 时 |x| 中位数 {s['final_x']:.2f} m")
print(f"  摆杆调节时间中位数 {s['settle']:.2f} s")


## 3.1 敏感性：初始扰动尺度 δ

对比无输入限幅与 $|u|\le 5$ N 时的摆杆平衡成功率。

In [ ]:

deltas = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
rng = np.random.default_rng(43)
draws = [rng.standard_normal(4) for _ in range(50)]

print(f"{'δ':>6} {'无饱和-摆杆':>12} {'5N-摆杆':>10} {'峰值推力中位数':>16}")
res = []
for d in deltas:
    x0s = [XSTAR + d * w for w in draws]
    sf = c_summary(c_run(x0s=x0s)[0])
    sl = c_summary(c_run(x0s=x0s, lim=5.0)[0])
    res.append((d, sf["pole_rate"], sl["pole_rate"], sf["peak"]))
    print(f"{d:6.2f} {sf['pole_rate']:12.0%} {sl['pole_rate']:10.0%} {sf['peak']:13.1f} N")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot([r[0] for r in res], [r[1] for r in res], "o-", label="无输入限幅")
ax.plot([r[0] for r in res], [r[2] for r in res], "s--", label="|u| ≤ 5 N")
ax.set_xlabel("扰动尺度 δ（x0 = x* + δ·N(0,I)）")
ax.set_ylabel("摆杆平衡成功率（N=50）")
ax.set_ylim(-0.02, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## 3.2 敏感性：LQR 权重 R 与 Q

观察控制代价对增益、峰值推力、摆杆平衡和小车回中的影响。

In [ ]:

rng = np.random.default_rng(42)
x0s = [XSTAR + 0.1 * rng.standard_normal(4) for _ in range(100)]

print("R 扫描（Q = diag(10,10,1,1)）：")
print(f"{'R':>7} {'‖D‖':>9} {'峰值推力':>10} {'摆杆成功率':>10} {'完整成功率':>10} {'5s时|x|':>10}")
r_rows = []
for R in [0.01, 0.1, 1.0, 10.0, 100.0]:
    c = c_design(R=np.array([R]))
    Dv = c.D().ravel()
    s = c_summary(c_run(R=np.array([R]), x0s=x0s)[0])
    r_rows.append((R, float(np.linalg.norm(Dv)), s))
    print(f"{R:7.2f} {np.linalg.norm(Dv):9.2f} {s['peak']:9.1f} N "
          f"{s['pole_rate']:10.0%} {s['rate']:10.0%} {s['final_x']:9.2f} m")

print("\nQ_theta 扫描（Q = diag(10,w,1,1), R = 1）：")
print(f"{'w':>7} {'‖D‖':>9} {'摆杆成功率':>10} {'峰值推力':>10}")
for w in [1.0, 10.0, 100.0]:
    Q = np.diag([10.0, w, 1.0, 1.0])
    s = c_summary(c_run(Q=Q, x0s=x0s)[0])
    print(f"{w:7.1f} {np.linalg.norm(c_design(Q=Q).D()):9.2f} "
          f"{s['pole_rate']:10.0%} {s['peak']:9.1f} N")

fig, ax = plt.subplots(figsize=(6, 3.6))
Rs = [r[0] for r in r_rows]
ax.plot(Rs, [r[1] for r in r_rows], "o-", label="‖D‖")
ax.set_xscale("log")
ax.set_xticks(Rs); ax.set_xticklabels(["0.01", "0.1", "1", "10", "100"])
ax.set_xlabel("R"); ax.set_ylabel("‖D‖"); ax.grid(alpha=0.3)
ax.legend(loc="upper right")
ax2 = ax.twinx()
ax2.plot(Rs, [r[2]["final_x"] for r in r_rows], "s--", color="tab:red", label="5 s 时 |x|")
ax2.set_ylabel("5 s 时 |x| 中位数 [m]", color="tab:red")
ax.set_title("R 增大：增益下降，但小车回中变慢")
plt.show()


## 3.3 敏感性：推力限幅

原代码没有输入饱和；这里人为加入 $|u|\le F$ 的限幅，观察约束对吸引域的影响。

In [ ]:

limits = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 1e6]
fl = []
print(f"{'限幅 [N]':>10} {'摆杆成功率':>10} {'完整成功率':>10} {'max|x| 中位数':>14}")
for lim in limits:
    s = c_summary(c_run(x0s=x0s, lim=lim)[0])
    fl.append(s)
    label = "∞" if lim > 1e5 else f"{lim:g}"
    print(f"{label:>10} {s['pole_rate']:10.0%} {s['rate']:10.0%} {s['max_x']:11.2f} m")

fig, ax = plt.subplots(figsize=(6, 3.6))
xs = np.arange(len(limits))
ax.plot(xs, [s["pole_rate"] for s in fl], "o-", label="摆杆平衡")
ax.plot(xs, [s["rate"] for s in fl], "s--", label="摆杆+小车回到容差")
ax.set_xticks(xs)
ax.set_xticklabels(["0.5", "1", "2", "5", "10", "20", "50", "∞"])
ax.set_xlabel("推力限幅 [N]（∞ = 无饱和）")
ax.set_ylabel("成功率（δ=0.1, N=100）")
ax.set_ylim(-0.02, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()


## 3.4 敏感性：物理参数 mc、mp、lc

利用 `MultibodyPlant` 的 context 参数直接修改小车质量、摆杆质量和质心距。

In [ ]:

rng = np.random.default_rng(42)
x0s = [XSTAR + 0.1 * rng.standard_normal(4) for _ in range(100)]

def c_param_sweep(name, values, key):
    print(f"\n{name} 扫描：")
    print(f"{name:>5} {'σ_min(C)':>10} {'cond(C)':>10} {'‖D‖':>8} {'峰值推力':>10} {'摆杆成功率':>10} {'5s时|x|':>9}")
    out = []
    for v in values:
        mods = dict(mc=MC, mp=MP, lc=LC); mods[key] = v
        ldv = c_linear(**mods)
        c = c_design(**mods)
        s = c_summary(c_run(**mods, x0s=x0s)[0])
        out.append((v, ldv, c, s))
        print(f"{v:5.2f} {ldv['sv'][-1]:10.4f} {ldv['sv'][0]/ldv['sv'][-1]:10.1f} "
              f"{np.linalg.norm(c.D()):8.1f} {s['peak']:8.1f} N {s['pole_rate']:10.0%} {s['final_x']:7.2f} m")
    return out

mc_rows = c_param_sweep("mc", [2.0, 5.0, 10.0, 20.0], "mc")
mp_rows = c_param_sweep("mp", [0.1, 0.5, 1.0, 2.0], "mp")
lc_rows = c_param_sweep("lc", [0.25, 0.5, 1.0], "lc")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
axs[0].plot([r[0] for r in mc_rows], [r[1]["sv"][-1] for r in mc_rows], "o-")
axs[0].set_xlabel("mc [kg]"); axs[0].set_ylabel("σ_min(C)")
axs[0].set_yscale("log"); axs[0].grid(alpha=0.3)
axs[0].set_title("小车质量 ↑ → 可控性最弱方向 ↓")
axs[1].plot([r[0] for r in mc_rows], [r[3]["peak"] for r in mc_rows], "s-", color="tab:red")
axs[1].set_xlabel("mc [kg]"); axs[1].set_ylabel("前 0.5 s 峰值推力中位数 [N]")
axs[1].grid(alpha=0.3); axs[1].set_title("小车质量 ↑ → 所需控制力 ↑")
plt.tight_layout()
plt.show()


## 4. 代表性轨迹：有/无推力限幅

同一初始状态，对比无限幅（成功）和 ±2 N 限幅（失败）。

In [ ]:

# 同一初始状态：无限幅（成功） vs |u|<=2 N（失败）
x0_case = XSTAR + 0.1 * np.array([0.5, 1.0, 0.2, 1.0])

fig, axs = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for lim, color, label in ((1e6, "tab:green", "无输入限幅"), (2.0, "tab:red", "|u| ≤ 2 N")):
    ctrl = c_design()
    diag = c_build(ctrl, lim=lim)
    ts, X, U, Uc = c_sim(diag, x0_case)
    axs[0, 0].plot(ts, X[0], color=color, label=label)
    axs[0, 1].plot(ts, X[1] - np.pi, color=color, label=label)
    axs[1, 0].plot(ts, X[3], color=color, label=label)
    axs[1, 1].plot(ts, U[0], color=color, label=label)

for ax, ti in zip(axs.ravel(), ["小车位置 x [m]", "θ - π（未 wrap）[rad]",
                                "θ̇ [rad/s]", "小车推力 u [N]"]):
    ax.set_title(ti); ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_xlabel("t [s]")
axs[0, 1].set_ylim(-2.0, 10.0)
fig.suptitle("Cart-Pole LQR：同一初始状态，有无推力限幅的对比")
fig.tight_layout()
plt.show()


## 结论速览

1. **可控性**：`rank(C)=4`，但开环有两个零特征值（小车位置是积分链）；奇异值为 `[4.326, 4.326, 0.0907, 0.0907]`，条件数约 48。两个小奇异值对应小车位置方向，说明它虽然可控但很“弱”。
2. **非最小相位**：$u\to x$ 的传递函数有一个右半平面零点 $+4.43$。为了把摆杆拉回，小车必须先向“错误方向”移动，因此 0.1 的初始扰动会引起约 0.3 m 的小车反向偏移。
3. **LQR**：$D=[3.16,-254.39,9.77,-55.18]$，闭环极点为 $-4.72,-4.57,-0.382\pm0.376j$。摆杆约 0.2 s 内回正；但小车位置对应一对慢极点（时间常数约 2.6 s），5 s 内小车往往还没完全回到原点。
4. **原代码没有推力饱和**：100 次 0.1 级随机扰动中，摆杆 **100%** 平衡，完整状态（含小车回中）98%；峰值推力中位数 17.2 N、90 分位 40 N、最大 64.6 N。对真实执行器来说这个力不算小。
5. **力限幅是决定性的**：δ=0.1 时，限幅 20 N → 摆杆成功率 96%，10 N → 64%，5 N → 42%，2 N → 27%。原 notebook 的“全部成功”依赖于无约束力。
6. **参数敏感性**：小车质量 mc 影响最大（2→20 kg 时峰值推力 5→32 N，小车回中变慢）；摆杆质量影响很小；R 增大能略降推力，但会显著拖慢小车回中，摆杆平衡不受影响；Q_theta 在 1–100 范围内几乎无影响。
7. **与讲义对照**：取 mc=mp=1、lc=1、g=1 时，A、B、零点、LQR 增益和闭环极点与课程笔记 §3.4.3/§3.4.8 完全一致。

> 完整表格与推导见 `Exp3'2-cartpole-作业解答.md`；批量脚本见 `exp3_2_analysis/cp_analysis.py`。
